# Notebook 5: Tail Risk Estimation
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sys; sys.path.insert(0,"../src")
import warnings; warnings.filterwarnings("ignore")
from scipy import stats
np.random.seed(42)
DARK,GRID,BORDER,TITLE,MUTED="#0d1117","#21262d","#30363d","#f0f6fc","#8b949e"
COLORS=["#58a6ff","#3fb950","#f78166","#d2a8ff","#ffa657","#79c0ff"]
plt.rcParams.update({"figure.facecolor":DARK,"axes.facecolor":DARK,"axes.edgecolor":BORDER,"text.color":"#c9d1d9","xtick.color":MUTED,"ytick.color":MUTED,"grid.color":GRID,"axes.titlecolor":TITLE,"font.size":10})
returns=pd.read_csv("../data/returns.csv",index_col="Date",parse_dates=True)
weights=np.array([0.25,0.25,0.20,0.15,0.15])
port=returns@weights; port_prices=(1+port).cumprod()*100
print(f"Loaded {len(returns):,} obs")

Loaded 2,609 obs


In [2]:
from tail_risk import compute_var_es
var_es=compute_var_es(port)
print(var_es.round(4))

               VaR      ES  ES/VaR
confidence                        
90%         0.0089  0.0132   1.484
95%         0.0115  0.0163   1.417
97%         0.0135  0.0202   1.491
99%         0.0168  0.0279   1.663
99%         0.0212  0.0362   1.710


In [3]:
fig,axes=plt.subplots(1,2,figsize=(14,5),facecolor=DARK)
ax1=axes[0]
ax1.hist(port*100,bins=100,density=True,color=COLORS[0],alpha=0.65)
x=np.linspace(port.min()*100,port.max()*100,300)
ax1.plot(x,stats.norm.pdf(x,port.mean()*100,port.std()*100),color=COLORS[4],linewidth=2,linestyle="--",label="Normal")
for cl,c in zip([0.95,0.99],[COLORS[4],COLORS[2]]):
    v=np.percentile(port*100,(1-cl)*100)
    ax1.axvline(v,color=c,linewidth=1.8,linestyle="--",label=f"{int(cl*100)}% VaR")
ax1.set_title("Return Distribution",color=TITLE); ax1.legend(fontsize=8)
ax1.set_facecolor(DARK); ax1.grid(True,color=GRID,alpha=0.5); ax1.tick_params(colors=MUTED)
ax2=axes[1]
rv99=port.rolling(252).quantile(0.01)*(-100)
rv95=port.rolling(252).quantile(0.05)*(-100)
ax2.plot(rv95.index,rv95,color=COLORS[4],linewidth=1.2,label="95% VaR")
ax2.plot(rv99.index,rv99,color=COLORS[2],linewidth=1.2,label="99% VaR")
ax2.set_title("Rolling Tail Risk",color=TITLE); ax2.legend(fontsize=8)
ax2.set_facecolor(DARK); ax2.grid(True,color=GRID,alpha=0.5); ax2.tick_params(colors=MUTED)
plt.tight_layout()
plt.savefig("../results/05_tail_risk_distribution.png",dpi=150,bbox_inches="tight",facecolor=DARK)
plt.show()